In [22]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from pathlib import Path

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

# SMOTE for class balancing
from imblearn.over_sampling import SMOTE

# Pipeline for combining SMOTE and model training
from imblearn.pipeline import Pipeline as ImbPipeline

# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [14]:
# Load final feature-engineered dataset

file_path = Path(
    "../data/processed/loan_approval_final_features.csv"
)

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")

print("\nDataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Loaded Successfully!

Dataset Shape: (32409, 17)

Column Names:
['num__person_income', 'num__loan_amnt', 'num__loan_int_rate', 'num__loan_percent_income', 'num__loan_to_income_ratio', 'cat__person_home_ownership_MORTGAGE', 'cat__person_home_ownership_OWN', 'cat__person_home_ownership_RENT', 'cat__loan_grade_A', 'cat__loan_grade_B', 'cat__loan_grade_D', 'cat__loan_grade_E', 'cat__loan_grade_F', 'cat__cb_person_default_on_file_N', 'cat__cb_person_default_on_file_Y', 'loan_status', 'dataset_split']

First 5 Rows:


,num__person_income,num__loan_amnt,num__loan_int_rate,num__loan_percent_income,num__loan_to_income_ratio,cat__person_home_ownership_MORTGAGE,cat__person_home_ownership_OWN,cat__person_home_ownership_RENT,cat__loan_grade_A,cat__loan_grade_B,cat__loan_grade_D,cat__loan_grade_E,cat__loan_grade_F,cat__cb_person_default_on_file_N,cat__cb_person_default_on_file_Y,loan_status,dataset_split
0,0.365413,0.066435,-0.336807,-0.471326,-0.495240,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,train
1,-0.300768,-0.171904,1.514924,-0.002815,-0.005785,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,train
2,0.460581,1.655355,-0.012511,0.465695,0.482449,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,train
3,0.436066,1.019787,-0.025483,0.090887,0.091062,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,train
4,-0.510139,0.066435,0.691212,0.840504,0.802078,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,train



Missing Values:
num__person_income                     0
num__loan_amnt                         0
num__loan_int_rate                     0
num__loan_percent_income               0
num__loan_to_income_ratio              0
cat__person_home_ownership_MORTGAGE    0
cat__person_home_ownership_OWN         0
cat__person_home_ownership_RENT        0
cat__loan_grade_A                      0
cat__loan_grade_B                      0
cat__loan_grade_D                      0
cat__loan_grade_E                      0
cat__loan_grade_F                      0
cat__cb_person_default_on_file_N       0
cat__cb_person_default_on_file_Y       0
loan_status                            0
dataset_split                          0
dtype: int64


In [15]:
# Separate training and testing datasets

train_df = df[df["dataset_split"] == "train"].copy()
test_df = df[df["dataset_split"] == "test"].copy()

# Separate input features and target variable

X_train = train_df.drop(columns=["loan_status", "dataset_split"])
y_train = train_df["loan_status"]

X_test = test_df.drop(columns=["loan_status", "dataset_split"])
y_test = test_df["loan_status"]

# Display dataset information

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nInput Feature Names:")
print(X_train.columns.tolist())

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Features: (25927, 15)
Testing Features: (6482, 15)
Training Target: (25927,)
Testing Target: (6482,)

Input Feature Names:
['num__person_income', 'num__loan_amnt', 'num__loan_int_rate', 'num__loan_percent_income', 'num__loan_to_income_ratio', 'cat__person_home_ownership_MORTGAGE', 'cat__person_home_ownership_OWN', 'cat__person_home_ownership_RENT', 'cat__loan_grade_A', 'cat__loan_grade_B', 'cat__loan_grade_D', 'cat__loan_grade_E', 'cat__loan_grade_F', 'cat__cb_person_default_on_file_N', 'cat__cb_person_default_on_file_Y']

Training Target Distribution:
loan_status
0    20257
1     5670
Name: count, dtype: int64

Testing Target Distribution:
loan_status
0    5064
1    1418
Name: count, dtype: int64


In [23]:
# Check class distribution before applying SMOTE

print("Training Class Distribution:")
print(y_train.value_counts())

print("\nTraining Class Percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Class Distribution:")
print(y_test.value_counts())

print("\nTraining Dataset Shape:", X_train.shape)
print("Testing Dataset Shape:", X_test.shape)

Training Class Distribution:
loan_status
0    20257
1     5670
Name: count, dtype: int64

Training Class Percentage:
loan_status
0    78.130906
1    21.869094
Name: proportion, dtype: float64

Testing Class Distribution:
loan_status
0    5064
1    1418
Name: count, dtype: int64

Training Dataset Shape: (25927, 15)
Testing Dataset Shape: (6482, 15)


In [24]:
# Logistic Regression - SMOTE + Training + Saving

# Create SMOTE and Logistic Regression pipeline
logistic_regression = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),

    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
logistic_regression.fit(X_train, y_train)

# Create models folder
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save trained model
model_path = models_dir / "logistic_regression.pkl"

joblib.dump(logistic_regression, model_path)

print("Logistic Regression with SMOTE Trained Successfully!")

print("\nModel Saved At:")
print(model_path.resolve())

print("\nFile Exists:", model_path.exists())

print("\nNumber of Input Features:")
print(logistic_regression.n_features_in_)

Logistic Regression with SMOTE Trained Successfully!

Model Saved At:
C:\Users\Vishnu SJ\Music\Edure final project\AI-Powered Loan Approval Prediction System\models\logistic_regression.pkl

File Exists: True

Number of Input Features:
15


In [25]:
# Decision Tree - SMOTE + Training + Saving

# Create SMOTE and Decision Tree pipeline
decision_tree = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),

    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

# Train the model
decision_tree.fit(X_train, y_train)

# Save trained model
model_path = models_dir / "decision_tree.pkl"

joblib.dump(decision_tree, model_path)

print("Decision Tree with SMOTE Trained Successfully!")

print("\nModel Saved At:")
print(model_path.resolve())

print("\nFile Exists:", model_path.exists())

print("\nNumber of Input Features:")
print(decision_tree.n_features_in_)

Decision Tree with SMOTE Trained Successfully!

Model Saved At:
C:\Users\Vishnu SJ\Music\Edure final project\AI-Powered Loan Approval Prediction System\models\decision_tree.pkl

File Exists: True

Number of Input Features:
15


In [26]:
# Random Forest - SMOTE + Training + Saving

# Create SMOTE and Random Forest pipeline
random_forest = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),

    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
random_forest.fit(X_train, y_train)

# Save trained model
model_path = models_dir / "random_forest.pkl"

joblib.dump(random_forest, model_path)

print("Random Forest with SMOTE Trained Successfully!")

print("\nModel Saved At:")
print(model_path.resolve())

print("\nFile Exists:", model_path.exists())

print("\nNumber of Input Features:")
print(random_forest.n_features_in_)

Random Forest with SMOTE Trained Successfully!

Model Saved At:
C:\Users\Vishnu SJ\Music\Edure final project\AI-Powered Loan Approval Prediction System\models\random_forest.pkl

File Exists: True

Number of Input Features:
15


In [27]:
# Gradient Boosting - SMOTE + Training + Saving

# Create SMOTE and Gradient Boosting pipeline
gradient_boosting = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),

    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the model
gradient_boosting.fit(X_train, y_train)

# Save trained model
model_path = models_dir / "gradient_boosting.pkl"

joblib.dump(gradient_boosting, model_path)

print("Gradient Boosting with SMOTE Trained Successfully!")

print("\nModel Saved At:")
print(model_path.resolve())

print("\nFile Exists:", model_path.exists())

print("\nNumber of Input Features:")
print(gradient_boosting.n_features_in_)

Gradient Boosting with SMOTE Trained Successfully!

Model Saved At:
C:\Users\Vishnu SJ\Music\Edure final project\AI-Powered Loan Approval Prediction System\models\gradient_boosting.pkl

File Exists: True

Number of Input Features:
15


In [28]:
from sklearn.ensemble import AdaBoostClassifier

In [29]:
# AdaBoost - SMOTE + Training + Saving

# Create SMOTE and AdaBoost pipeline
ada_boost = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),

    ("classifier", AdaBoostClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the model
ada_boost.fit(X_train, y_train)

# Save trained model
model_path = models_dir / "ada_boost.pkl"

joblib.dump(ada_boost, model_path)

print("AdaBoost with SMOTE Trained Successfully!")

print("\nModel Saved At:")
print(model_path.resolve())

print("\nFile Exists:", model_path.exists())

print("\nNumber of Input Features:")
print(ada_boost.n_features_in_)

AdaBoost with SMOTE Trained Successfully!

Model Saved At:
C:\Users\Vishnu SJ\Music\Edure final project\AI-Powered Loan Approval Prediction System\models\ada_boost.pkl

File Exists: True

Number of Input Features:
15
